# 32 · Gold: fact tables
**Owners: R (`fact_event_loss`) · M (`fact_state_year`)** · Layer: gold · Runs after: 23, 24, 30, 31

| Fact | Type | Grain |
|---|---|---|
| `fact_event_loss` | Transaction | One row per in-scope ICA catastrophe event (natural hazard, starting 2011 or later), attributed to its primary state |
| `fact_state_year` | Periodic snapshot | One row per state per year, 2011–2026 |

**Why two facts:** population and income describe a state in a year, not an event. Keeping them separate stops anyone summing population across events. Both facts share `dim_location` and `dim_date` (conformed dimensions), so they can be combined by state and year.

**Key mapping:** every business value is swapped for a surrogate key. Anything that doesn't match a dimension gets −1 (Unknown), never dropped.

In [0]:
%run ./00_common

In [0]:
ev = read("silver", "ica_events").where("in_scope")
rows_in = ev.count()
de = read("silver", "disaster_events").select("de_id", "deaths", "injuries", "evacuated", "homes_destroyed")
match = read("silver", "ica_de_match").where("match_confidence <> 'low'").select("event_id", "de_id")   # low-confidence matches don't enrich
d_event = read("gold", "dim_event").select("event_key", "event_id")
d_loc = read("gold", "dim_location").select("location_key", "state_code")
d_hz = read("gold", "dim_hazard").select("hazard_key", "hazard_group")
d_date = read("gold", "dim_date").where("date_key > 0").select("date_key", "full_date")
CLAIMS = ["total_claims", "domestic_building_claims", "domestic_content_claims", "domestic_motor_claims", "domestic_other_claims",
          "commercial_property_claims", "commercial_motor_claims", "commercial_bi_claims", "commercial_other_claims", "commercial_crop_claims"]

fact = (ev.join(d_event, "event_id", "left")
          .join(d_loc, "state_code", "left")
          .join(d_hz, "hazard_group", "left")
          .join(d_date.withColumnRenamed("date_key", "start_date_key"), F.col("start_date") == F.col("full_date"), "left").drop("full_date")
          .join(d_date.withColumnRenamed("date_key", "end_date_key"), F.col("end_date") == F.col("full_date"), "left").drop("full_date")
          .join(match, "event_id", "left").join(de, "de_id", "left")
          .select(F.col("event_key").cast("int"),
                  F.coalesce("start_date_key", F.lit(-1)).alias("start_date_key"),
                  F.coalesce("end_date_key", F.lit(-1)).alias("end_date_key"),
                  F.coalesce("location_key", F.lit(-1)).cast("int").alias("location_key"),
                  F.coalesce("hazard_key", F.lit(-1)).cast("int").alias("hazard_key"),
                  "original_loss_aud", "normalised_loss_2022_aud", *CLAIMS, "duration_days",
                  F.col("deaths").cast("int"), F.col("injuries").cast("int"), F.col("evacuated").cast("int"),
                  F.col("homes_destroyed").cast("int"), F.lit(RUN_ID).alias("run_id")))
n = write(fact, "gold", "fact_event_loss")
log_step("gold", "load_fact_event_loss", "gold.fact_event_loss", rows_in, n, 0)

### `fact_state_year`: nearest-year fill (DQ6)
ABS has population only for 2020–2025 and median income only for 2019–2023. For every state and year from 2011 to 2026, take the value from the **nearest year that has one** (the earlier year wins a tie). The year actually used and an `imputed` flag are stored beside each value, so nobody mistakes a borrowed value for a measured one.

In [0]:
sy = read("silver", "abs_state_year")
grid = (d_loc.where("location_key > 0")
        .crossJoin(spark.sql(f"SELECT explode(sequence(2011, {SCOPE_END_YEAR})) AS year")))

def nearest(value_col):
    avail = sy.where(F.col(value_col).isNotNull()).select("state_code", F.col("year").alias("src_year"), value_col)
    w = Window.partitionBy("location_key", "year").orderBy(F.abs(F.col("src_year") - F.col("year")), "src_year")
    return (grid.join(avail, "state_code").withColumn("rn", F.row_number().over(w)).where("rn = 1")
            .select("location_key", "year", value_col, F.col("src_year").alias(f"{value_col}_year_used")))

snap = (grid.select("location_key", "year")
        .join(nearest("population"), ["location_key", "year"], "left")
        .join(nearest("median_income_aud"), ["location_key", "year"], "left")
        .select(F.col("location_key").cast("int"), (F.col("year") * 10000 + 630).alias("date_key"),           # 30 June: ABS reference date
                F.col("population").cast("bigint"), F.col("population_year_used").cast("int"),
                (F.col("population_year_used") != F.col("year")).alias("population_imputed"),
                "median_income_aud", F.col("median_income_aud_year_used").cast("int").alias("income_year_used"),
                (F.col("median_income_aud_year_used") != F.col("year")).alias("income_imputed"),
                F.lit(RUN_ID).alias("run_id")))
n_sy = write(snap, "gold", "fact_state_year")
log_step("gold", "load_fact_state_year", "gold.fact_state_year", grid.count(), n_sy, 0,
         f"{snap.where('population_imputed').count()} population and {snap.where('income_imputed').count()} income values imputed")

### Informational constraints (Databricks only)

In [0]:
add_constraints([
    *[f"ALTER TABLE {tname('gold', t)} ALTER COLUMN {k} SET NOT NULL" for t, k in
      [("dim_date", "date_key"), ("dim_location", "location_key"), ("dim_hazard", "hazard_key"), ("dim_event", "event_key")]],
    f"ALTER TABLE {tname('gold', 'dim_date')} ADD CONSTRAINT pk_dim_date PRIMARY KEY (date_key)",
    f"ALTER TABLE {tname('gold', 'dim_location')} ADD CONSTRAINT pk_dim_location PRIMARY KEY (location_key)",
    f"ALTER TABLE {tname('gold', 'dim_hazard')} ADD CONSTRAINT pk_dim_hazard PRIMARY KEY (hazard_key)",
    f"ALTER TABLE {tname('gold', 'dim_event')} ADD CONSTRAINT pk_dim_event PRIMARY KEY (event_key)",
    f"ALTER TABLE {tname('gold', 'fact_event_loss')} ADD CONSTRAINT fk_fel_event FOREIGN KEY (event_key) REFERENCES {tname('gold', 'dim_event')}",
    f"ALTER TABLE {tname('gold', 'fact_event_loss')} ADD CONSTRAINT fk_fel_start FOREIGN KEY (start_date_key) REFERENCES {tname('gold', 'dim_date')}",
    f"ALTER TABLE {tname('gold', 'fact_event_loss')} ADD CONSTRAINT fk_fel_end FOREIGN KEY (end_date_key) REFERENCES {tname('gold', 'dim_date')}",
    f"ALTER TABLE {tname('gold', 'fact_event_loss')} ADD CONSTRAINT fk_fel_loc FOREIGN KEY (location_key) REFERENCES {tname('gold', 'dim_location')}",
    f"ALTER TABLE {tname('gold', 'fact_event_loss')} ADD CONSTRAINT fk_fel_haz FOREIGN KEY (hazard_key) REFERENCES {tname('gold', 'dim_hazard')}",
    f"ALTER TABLE {tname('gold', 'fact_state_year')} ADD CONSTRAINT fk_fsy_loc FOREIGN KEY (location_key) REFERENCES {tname('gold', 'dim_location')}",
    f"ALTER TABLE {tname('gold', 'fact_state_year')} ADD CONSTRAINT fk_fsy_date FOREIGN KEY (date_key) REFERENCES {tname('gold', 'dim_date')}",
])

In [0]:
display(read("gold", "fact_state_year").where("location_key = 1").orderBy("date_key"))